## Usando arquivo .env para controlar variaveis de ambiente
Para evitar exposição da chave `OPENAI_API_KEY` optei por utilizar arquivo `.env` com a informação da chave.

Para seguir o mesmo método basta criar um arquivo `.env` no mesmo diretório do arquivo `parent_rag.ipynb`.
A importação da chave será feita através da célula abaixo que faz a instalação de um biblioteca para carregar
os valores do arquivo `.env`.

In [12]:
%pip install python-dotenv
import dotenv
%load_ext dotenv
%dotenv

Note: you may need to restart the kernel to use updated packages.
The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.storage import InMemoryByteStore
from langchain.retrievers import ParentDocumentRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# Perguntas
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

In [ ]:
# Inicializando o modelo de embeddings
embeddings_model = OpenAIEmbeddings()

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    max_tokens=200,
)

In [ ]:
# Caminho do arquivo PDF
pdf_path = "os-sertoes.pdf"

load_pdf = PyPDFLoader(pdf_path, extract_images=False)

pages = load_pdf.load_and_split()

In [ ]:

# Configuração do divisor
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,     
    chunk_overlap=200,  
    length_function=len,      
    add_start_index=True    
)


In [ ]:
# Inicializando o armazenamento
store = InMemoryByteStore()

vectorstore = Chroma(embedding_function=embeddings_model)

In [ ]:
# Configurando o recuperador de documentos 
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,      
    docstore=store,               
    child_splitter=child_splitter,
    parent_splitter=parent_splitter 
)

# Adicionando documentos ao recuperador
parent_document_retriever.add_documents(pages, ids=None)

In [ ]:
# Definindo o template de prompt
TEMPLATE = """
Você é um especialista em literatura brasileira. Responda a pergunta abaixo utilizando o contexto informado:

Contexto: {context}

Pergunta: {question}
"""

prompt = PromptTemplate(input_variables=["context", "question"], template=TEMPLATE)

parser = StrOutputParser()

sequence = RunnableSequence(prompt | llm | parser)


In [ ]:
# Função para responder a uma pergunta
def answer_question(question: str):

    context = parent_document_retriever.invoke(question)

    response = sequence.invoke({"context": context, "question": question})
  
    return response

In [ ]:
# Iterando sobre as perguntas
for index, question in enumerate(questions):

    resposta = answer_question(question)

    print({"numero": index, "pergunta": question, "resposta": resposta})

{'numero': 0, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como uma região árida e hostil, marcada por extensos plainos ondulados e serranias. Ele destaca a dureza da natureza, com afloramentos gnáissicos e planaltos que se dobram, dificultando a vida dos habitantes locais. Essas características influenciam diretamente a vida dos habitantes, tornando a região desafiadora para a sobrevivência, com caudais de rios que revelam um pendor insensível para o norte e a presença de formações primitivas desaparecendo sob complexas séries de xistos metamórficos. Euclides da Cunha destaca a imponência dos quadros naturais do sertão nordestino, ressaltando como essas características influenc'}
{'numero': 1, 'pergunta': 'Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele r